In [1]:
import hickle
import pandas as pd
import scipy.signal
# import shap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# from astropy.table import Table
from scipy import ndimage
from tqdm import tqdm
import corner
from PIL import Image
import io
import os
import glob
import gc

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
import torchvision.transforms as T

from scipy.stats import chi2, spearmanr
from sklearn.datasets import make_s_curve
from sklearn.metrics import r2_score
import matplotlib as mpl

import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PosteriorCoverage, PlotSinglePosterior

from datasets import load_dataset, DatasetDict
from transformers import ViTFeatureExtractor, ViTForImageClassification, ViTModel, AdamW, AutoImageProcessor, AutoModel

In [2]:
GSWLC_data = load_dataset('juanpabloalfonzo/GSWLC_Data', split='train')
gswlc_data = GSWLC_data.to_pandas()

timescales_data_cami_fixed = np.load('/home/juanpabloalfonzo/Documents/ViT Morph Fitting/all_timescales_cami_fixed_v2.npy', allow_pickle=True).item()

In [3]:
temp_fuv = np.array(gswlc_data['FUV']).copy()
temp_fuv[temp_fuv > 25] = 25 + 0.3 * np.random.normal(size=(np.sum(temp_fuv > 25),))

temp_w2 = np.array(gswlc_data['W2']).copy()
temp_w2[temp_w2 > 23] = 23

temp_w3 = np.array(gswlc_data['W3']).copy()
temp_w3[temp_w3 > 23] = 23 + 0.3 * np.random.normal(size=(np.sum(temp_w3 > 23),))

temp_w4 = np.array(gswlc_data['W4']).copy()
temp_w4[temp_w4 > 23] = 23 + 0.3 * np.random.normal(size=(np.sum(temp_w4 > 23),))

eleven_bands_SED = np.array((
    temp_fuv, 
    gswlc_data['NUV'],
    gswlc_data['u'],
    gswlc_data['g'],
    gswlc_data['r'],
    gswlc_data['i'],
    gswlc_data['z'],
    gswlc_data['W1'],
    temp_w2,
    temp_w3,
    temp_w4,
)).T

tempt50 = np.array(gswlc_data['t_50']).copy()
tempt50[(tempt50 > 13.7)] = 13.7

tempt10 = np.array(gswlc_data['t_90']).copy()
tempt10[(tempt10 > 13.7)] = 13.7

tempt90 = np.array(gswlc_data['t_90']).copy()
tempt90[(tempt90 > 13.7)] = 13.7

tempssfr = np.array(gswlc_data['logSSFR_50']).copy()
tempssfr[tempssfr < -3.5] = tempssfr[tempssfr < -3.5] + 0.3 * np.random.normal(size=(np.sum(tempssfr < -3.5),))

tempmet = np.array(gswlc_data['logOH_50']).copy()
tempmet[tempmet < 7.1] = tempmet[tempmet < 7.1] + 0.1 * np.random.normal(size=(np.sum(tempmet < 7.1),))

theta = np.vstack((
    gswlc_data['logM_50_x'], 
    tempssfr,
    gswlc_data['specz'],
    tempt50,
    gswlc_data['tauV_50'],
    tempmet
)).T
theta[np.isnan(theta)] = 0.

five_bands_SED = np.array((
    gswlc_data['u'],
    gswlc_data['g'],
    gswlc_data['r'],
    gswlc_data['i'],
    gswlc_data['z'],
)).T

def safe_log_sfr(arr):
    return np.log10(np.clip(arr, 1e-3, None))

np.random.seed(256)
theta_full_sfh_sfr = np.vstack((
    gswlc_data['logM_50_x'], 
    tempssfr,
    gswlc_data['specz'],
    tempt50,
    gswlc_data['tauV_50'],
    tempmet,
    tempt10,
    tempt90,
    safe_log_sfr(timescales_data_cami_fixed['now']['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.1]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.3]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.7]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[0.8]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.3]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[1.7]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[2.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[2.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[3.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[3.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[4.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[4.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[5.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[5.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[6.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[6.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[7.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[7.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[8.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[8.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[9.0]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[9.5]['sfr']),
    safe_log_sfr(timescales_data_cami_fixed[10.0]['sfr'])
)).T

theta_full_sfh_sfr[np.isnan(theta_full_sfh_sfr)] = 0.
theta_full_sfh_sfr[:, 8:][theta_full_sfh_sfr[:, 8:] < -2.5] = theta_full_sfh_sfr[:, 8:][theta_full_sfh_sfr[:, 8:] < -2.5] + 0.3 * np.random.normal(size=(np.sum(theta_full_sfh_sfr[:, 8:] < -2.5),))

In [4]:
def train_ili_post_ensemble(x, theta, hidden_features=128, num_transforms=10, arch_type='original'):
    loader = NumpyLoader(x=x, theta=theta)

    theta_t = torch.as_tensor(theta, dtype=torch.float32, device=device)
    d_min = torch.amin(theta_t, dim=0)
    d_max = torch.amax(theta_t, dim=0)
    
    spread = d_max - d_min
    margin = spread * 0.2
    margin[margin == 0] = 1e-5 

    lower_bounds = d_min - margin
    upper_bounds = d_max + margin

    print("Lower:", lower_bounds[:5])
    print("Upper:", upper_bounds[:5])

    prior = ili.utils.Uniform(low=lower_bounds, high=upper_bounds, device=device)

    if arch_type == 'original':
        print("using original architecture")
        nets = [
                ili.utils.load_nde_sbi(engine='NPE', model='maf', hidden_features=50, num_transforms=5),
                ili.utils.load_nde_sbi(engine='NPE', model='mdn', hidden_features=50, num_components=6)
            ]
    else:  # Default 'maf_nsf' for v2 and ViT models
        print("using complex architecture")
        nets = [
            ili.utils.load_nde_sbi(engine='NPE', model='maf', hidden_features=hidden_features, num_transforms=num_transforms),
            ili.utils.load_nde_sbi(engine='NPE', model='nsf', hidden_features=hidden_features, num_transforms=num_transforms)
        ]

    train_args = {
        'training_batch_size': 64,
        'learning_rate': 1e-4,
        'max_num_epochs': 200,
        'stop_after_epochs': 20
    }

    runner = InferenceRunner.load(
        backend='sbi',
        engine='NPE',
        prior=prior,
        nets=nets,
        device=device,
        train_args=train_args,
        proposal=None,
        out_dir=None
    )

    posterior_ensemble, summaries = runner(loader=loader)
    return posterior_ensemble, summaries


def check_degeneracy(samples, n_params_limit=6):
    """Computes Spearman correlation on the first 6 physical parameters for speed."""
    n_p = min(samples.shape[1], n_params_limit)
    covs = np.zeros((n_p, n_p))
    pvals = np.ones((n_p, n_p))
    for i in range(n_p):
        for j in range(i):
            covs[i, j], pvals[i, j] = spearmanr(samples[:, i], samples[:, j])

    return covs[1:, :-1], pvals[1:, :-1]

In [5]:
# ==========================================
# SELECT MODALITY HERE
# Options: '5_band', '11_band', 'image', 'combined_5_band', 'combined_11_band', 'g_band', 'half_size', 'quarter_size'
# ==========================================
MODALITY = 'image' 

# Base directories
BASE_DIR = ''
EMBED_DIR = os.path.join(BASE_DIR, 'ViT_embeds')
MODEL_DIR = os.path.join(BASE_DIR, 'ili_models_large_arch_clean')
SAMPLES_DIR = os.path.join(BASE_DIR, 'samples_and_covs_large_arch_clean')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

train_index = int(0.8 * len(gswlc_data))


# Central Registry mapping modalities to filenames, embeddings, and network sizes
MODALITY_CONFIGS = {
    '5_band': {
        'embed_file': None,
        'sed_data': five_bands_SED,
        'model_file': 'posterior_ensemble_5_band_full_sfh_sfr_v2_full_train.hickle',
        'samples_file': '5_band_sfh_v2.hickle',
        'arch_type': 'complex'
    },
    '11_band': {
        'embed_file': None,
        'sed_data': eleven_bands_SED,
        'model_file': 'posterior_ensemble_11_band_full_sfh_sfr_v2_full_train.hickle',
        'samples_file': '11_band_sfh_v2.hickle',
        'arch_type': 'complex'
    },
    'image': {
        'embed_file': 'finetuned_embeds.hickle',
        'sed_data': None,
        'model_file': 'posterior_ensemble_image_full_sfh_sfr_v2_full_train.hickle',
        'samples_file': 'all_samples_and_covs_image_full_sfh_sfr_v2_full_sample.hickle',
        'arch_type': 'complex'
    },
    'combined_5_band': {
        'embed_file': 'finetuned_embeds.hickle',
        'sed_data': five_bands_SED,
        'model_file': 'posterior_ensemble_combined_5_band_full_sfh_sfr_v2_full_train.hickle',
        'samples_file': 'combined_5_band_sfh_v2.hickle',
        'arch_type': 'complex'
    },
    'combined_11_band': {
        'embed_file': 'finetuned_embeds.hickle',
        'sed_data': eleven_bands_SED,
        'model_file': 'posterior_ensemble_combined_11_band_full_sfh_sfr_v2_full_train.hickle',
        'samples_file': 'combined_11_band_sfh_v2.hickle',
        'arch_type': 'complex'
    },
    'g_band': {
        'embed_file': 'g_band_embeds.hickle',
        'sed_data': None,
        'model_file': 'posterior_ensemble_g_band_image_full_train.hickle',
        'samples_file': 'g_band_image.hickle',
        'arch_type': 'complex'
    },
    'half_size': {
        'embed_file': 'half_size_embeds.hickle',
        'sed_data': None,
        'model_file': 'posterior_ensemble_half_size_image_full_train.hickle',
        'samples_file': 'half_size_image.hickle',
        'arch_type': 'complex'
    },
    'quarter_size': {
        'embed_file': 'quarter_size_embeds.hickle',
        'sed_data': None,
        'model_file': 'posterior_ensemble_quarter_size_image_full_train.hickle',
        'samples_file': 'quarter_size_image.hickle',
        'arch_type': 'complex'
    }
}

cfg = MODALITY_CONFIGS[MODALITY]

# Dynamically construct input array x_data
embeds = None
if cfg['embed_file'] is not None:
    embed_path = os.path.join(EMBED_DIR, cfg['embed_file'])
    print(f"Loading embeddings from: {embed_path}")
    embeds = np.array(hickle.load(embed_path))

if embeds is not None and cfg['sed_data'] is not None:
    x_data = np.hstack((embeds, cfg['sed_data']))
elif embeds is not None:
    x_data = embeds
else:
    x_data = cfg['sed_data']

# Define all dynamic output paths for this modality
model_save_path = os.path.join(MODEL_DIR, cfg['model_file'])
final_samples_path = os.path.join(SAMPLES_DIR, cfg['samples_file'])
temp_chunk_dir = os.path.join(SAMPLES_DIR, f'chunks_{MODALITY}')

print(f"--- Modality: {MODALITY} ---")
print(f"Input features shape : {x_data.shape}")
print(f"Model save path      : {model_save_path}")
print(f"Chunk temp dir       : {temp_chunk_dir}")
print(f"Final samples path   : {final_samples_path}")

In [6]:
posterior_ensemble, summaries = train_ili_post_ensemble(
    x_data[:train_index], 
    theta_full_sfh_sfr[:train_index],
    arch_type=cfg['arch_type']
)

hickle.dump([posterior_ensemble, summaries], model_save_path)
print(f"Saved trained ensemble to {model_save_path}")

In [ ]:
CHUNK_SIZE = 5000  
os.makedirs(temp_chunk_dir, exist_ok=True)

# Load trained model for the active modality
posterior_ensemble, summaries = hickle.load(model_save_path)

def process_chunk(start_idx, end_idx, chunk_file_path, input_data, ensemble):
    """Samples posterior and computes 5x5 covariance for a slice of galaxies."""
    local_samples = []
    local_covs = []
    
    for i in range(start_idx, end_idx):
        xobs = input_data[i]
        posterior_samples = ensemble.sample(
            (1000,), torch.as_tensor(xobs, dtype=torch.float32, device=device), show_progress_bars=False
        )
        
        numpy_samples = posterior_samples.detach().cpu().numpy()
        local_samples.append(numpy_samples)
        
        covs, _ = check_degeneracy(numpy_samples)
        local_covs.append(covs)
        
    hickle.dump([local_covs, local_samples], chunk_file_path)
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Calculate total chunks (including remainder if any)
total_items = len(x_data)
num_chunks = (total_items + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Starting batched processing for '{MODALITY}' ({total_items} galaxies across {num_chunks} chunks)...")

for chunk_idx in tqdm(range(num_chunks), desc=f"Sampling {MODALITY}"):
    chunk_file = os.path.join(temp_chunk_dir, f'chunk_{chunk_idx:04d}.hickle')
    
    # Safely skip chunks that already exist on disk
    if os.path.exists(chunk_file):
        continue
        
    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, total_items)
    
    process_chunk(start_idx, end_idx, chunk_file, x_data, posterior_ensemble)
    gc.collect()

# Merge chunks into final modality file
print(f"Merging all chunks into {final_samples_path}...")
final_samples = []
final_covs = []

chunk_files = sorted(glob.glob(os.path.join(temp_chunk_dir, 'chunk_*.hickle')))

for f in tqdm(chunk_files, desc="Loading chunks"):
    chunk_covs, chunk_samples = hickle.load(f)
    final_covs.extend(chunk_covs)
    final_samples.extend(chunk_samples)

hickle.dump([final_covs, final_samples], final_samples_path)
print(f"Merge complete for {MODALITY}! Final list length: {len(final_samples)}")